<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_06_model_training/seq2one/stage_06_02_ridge_seq2one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_06_02 - SEQ2ONE - Modelo Ridge**

**Introducción**

Luego de establecer el baseline Naive, se introduce **Ridge Regression** como
primer modelo **entrenable** del stage_07 bajo un enfoque **seq2one**.

Ridge es una regresión lineal con **regularización L2**, diseñada para manejar
vectores de alta dimensión y features correlacionadas, manteniendo un control
explícito de la complejidad del modelo.

En este proyecto, Ridge permite evaluar cuánto valor puede capturarse mediante
una relación lineal directa entre la ventana histórica intradía
(L x N → cantidad de features) y el target escalar futuro.

---

**Rol en el pipeline**

- Primer modelo con capacidad de aprendizaje real.
- Referencia lineal fuerte y estable.
- Punto de comparación obligatorio para modelos no lineales posteriores.
- Si modelos más complejos no superan a Ridge en VALID, su aporte es cuestionable.

---

**Esquema general**

- **Entrada (X):** vector aplanado de LxN features.
- **Salida (Y):** valor escalar futuro.
- **Entrenamiento:** conjunto TRAIN.
- **Evaluación:** conjunto VALID.
- **Regularización:** L2 (controlada por el parámetro \(\alpha\)).

Las métricas obtenidas se almacenan como artefactos y se utilizan posteriormente en el **stage_07** para la comparación final entre modelos.


## **1. Imports + paths**

In [2]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

## **2. Acceso a drive**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## **3. Rutas de ventanas seq2one y scalers**

In [4]:
from pathlib import Path
import os

WINDOWS_SEQ2ONE_DIR = Path(
    os.environ.get("WINDOWS_SEQ2ONE_DIR", "data/windows/seq2one/")
)

SCALERS_DIR = Path(
    os.environ.get("SCALERS_DIR", "data/scaled/")
)

window_sizes = [30, 60, 90, 120, 180]
targets = ['delta_60', 'delta_90', 'ret_60', 'ret_90']
splits = ['train', 'valid', 'test']

In [5]:
windows_paths = {}

for w in window_sizes:
    windows_paths[w] = {}

    for t in targets:
        windows_paths[w][t] = {}

        for s in splits:
            path = (
                DRIVE_DIR
                / WINDOWS_SEQ2ONE_DIR
                / f"L{w}"
                / f"windows_{t}_{s}.npz"
            )

            windows_paths[w][t][s] = path

#display(windows_paths)

#Como llamarlo:
#path_train_L60_delta = windows_paths[60]['delta_90']['train']
#print(path_train_L60_delta)

In [6]:
scalers_paths = {}
for t in targets:
  scalers_paths[t] = {}
  path = (
                DRIVE_DIR
                / SCALERS_DIR
                / f"scaler_{t}.pkl"
            )

  scalers_paths[t] = path

display(scalers_paths)

{'delta_60': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_60.pkl'),
 'delta_90': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_90.pkl'),
 'ret_60': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_ret_60.pkl'),
 'ret_90': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_ret_90.pkl')}

## **4. Reproducibilidad**

In [7]:
def set_seeds(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

## **5. Importar métricas comunes desde .py**

In [8]:
# 1) Define el root del proyecto (DEBE existir en esta misma celda)
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# 2) Agrega el root al PYTHONPATH (antes de importar)
if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))  # insert(0) para priorizarlo

# 3) Asegura que metrics sea paquete Python
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

# 4) Invalida cachés de import (importante en Colab)
importlib.invalidate_caches()

# 5) Diagnóstico (le muestra qué ve Python)
#print("DRIVE_DIR:", DRIVE_DIR)
#print("sys.path[0]:", sys.path[0])
#print("metrics exists:", (DRIVE_DIR / "metrics").exists())
#print("metrics __init__:", (DRIVE_DIR / "metrics" / "__init__.py").exists())

# 6) Imports reales
from metrics.seq2one_metrics import compute_seq2one_metrics
#from metrics.opportunity_filter_metrics import compute_opportunity_filter_metrics

print("OK - imports metrics.*")


OK - imports metrics.*


In [9]:
print(compute_seq2one_metrics.__doc__)


    Calcula métricas simples y comparables para modelos seq2one.

    Parámetros
    ----------
    y_true : np.ndarray
        Valores reales con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    y_pred : np.ndarray
        Valores predichos con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    compute_r2 : bool
        Si True, calcula R² sobre el vector completo.
    da_ignore_zeros : bool
        Si True, ignora casos donde el signo sea 0 en y_true o y_pred al calcular DA.
    allow_seq_inputs_take_last : bool
        Si True, permite inputs 2D (n_samples, seq_len) y toma el último paso [:, -1].
        Útil si algún modelo devuelve secuencia pero usted lo evalúa como many-to-one.

    Retorna
    -------
    metrics : dict
        Diccionario con métricas globales.
    


## **6. Carga de ventanas**

In [10]:
# --------------------------------------------------
# Función común: carga .npz estándar (X, y)
# --------------------------------------------------
def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar.

    Espera claves:
    - 'X': array (n_samples, seq_len, n_features)
    - 'y' o 'Y': array (n_samples, seq_len) o (n_samples, seq_len, 1)
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # Carga el NPZ (lectura).
    data = np.load(path)

    # Lee X (obligatoria).
    if "X" not in data:
        raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")
    X = data["X"]

    # Lee y: soporta 'y' (convención usada) o 'Y' (por compatibilidad).
    if "y" in data:
        y = data["y"]
    elif "Y" in data:
        y = data["Y"]
    else:
        raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

    # Devuelve X e y.
    return X, y

In [11]:
# --------------------------------------------------
# Función común: carga scaler .pkl (sklearn)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib (ej. StandardScaler/MinMaxScaler).
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # Carga el objeto scaler.
    scaler = joblib.load(path)

    # Devuelve scaler (tipo genérico).
    return scaler

In [12]:
from typing import Any, Dict, Mapping
from pathlib import Path

# --------------------------------------------------
# Carga completa: ventanas + scaler por window_size y target
# --------------------------------------------------
def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scalers_path: Mapping[str, Path],
) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler correspondiente
    a un (window_size, target).

    windows_paths[L][target][split] -> Path
    scalers_path[target] -> Path
    """

    # --------------------------
    # 1) Validaciones
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    if target not in scalers_path:
        raise KeyError(f"target='{target}' no existe en scalers_path")

    # --------------------------
    # 2) Paths
    # --------------------------
    train_path = windows_paths[window_size][target]["train"]
    valid_path = windows_paths[window_size][target]["valid"]
    test_path  = windows_paths[window_size][target]["test"]
    scaler_path = scalers_paths[target]

    # --------------------------
    # 3) Carga
    # --------------------------
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test,  y_test  = load_npz_windows(test_path)

    scaler = load_scaler(scaler_path)

    # --------------------------
    # 4) Inferir horizonte
    # --------------------------
    horizon = int(target.split("_")[-1])

    # --------------------------
    # 5) Retorno
    # --------------------------
    return {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {"X": X_train, "y": y_train},
        "valid": {"X": X_valid, "y": y_valid},
        "test":  {"X": X_test,  "y": y_test},
    }

In [13]:
import numpy as np

def maybe_flatten_X(X: np.ndarray, *, flatten: bool) -> np.ndarray:
    """
    Si flatten=True y X es 3D (N,L,F) -> (N, L*F)
    Si flatten=False -> retorna X tal cual.
    """
    if not flatten:
        return X
    if X.ndim == 3:
        N, L, F = X.shape
        return X.reshape(N, L * F)
    if X.ndim == 2:
        return X
    raise ValueError(f"X debe ser 2D o 3D, recibí shape={X.shape}")

In [14]:
def create_bundles(window_size, targets: list, windows_paths=windows_paths, scalers_paths=scalers_paths, *, flatten_X=False):

    bundles = []
    for t in targets:
        b = load_windows_and_scaler(
            window_size=window_size,
            target=t,
            windows_paths=windows_paths,
            scalers_path=scalers_paths,
        )
        if flatten_X:
            b["train"]["X"] = maybe_flatten_X(b["train"]["X"], flatten=True)
            b["valid"]["X"] = maybe_flatten_X(b["valid"]["X"], flatten=True)
            b["test"]["X"]  = maybe_flatten_X(b["test"]["X"],  flatten=True)
        bundles.append(b)

    # prints (opcional)
    for b in bundles:
        print(f"H{b['horizon']} Train:", b["train"]["X"].shape, b["train"]["y"].shape)
        print(f"H{b['horizon']} Valid:", b["valid"]["X"].shape, b["valid"]["y"].shape)
        print(f"H{b['horizon']} Test :", b["test"]["X"].shape,  b["test"]["y"].shape)
        print(f"Scaler H{b['horizon']}:", type(b["scaler"]).__name__)

    return tuple(bundles)


In [15]:
#bundle_delta_60, bundle_delta_90 = create_bundles(window_size = 30, targets = ['delta_60', 'delta_90'], windows_paths = windows_paths, scalers_paths = scalers_paths, flatten_X = False)
#bundle_ret_60, bundle_ret_90 = create_bundles(window_size = 30, targets = ['ret_60', 'ret_90'], windows_paths = windows_paths, scalers_paths = scalers_paths)

NOTA IMPORTANTE: COMO ACCEDER A LAS VENTANAS

Para el horizonte: `h`

  - Train
    - `X`: `bundle_h["train"]["X"]`
    - `y`: `bundle_h["train"]["y"]`

  - Valid
    - `X`: `bundle_h["valid"]["X"]`
    - `y`:`bundle_h["valid"]["y"]`

  - Test
    - `X`: `bundle_h["test"]["X"]`
    - `y`: `bundle_h["test"]["y"]`

  - Scaler
    - `bundle_h["scaler"]`

## **7. Sanity Check**

In [16]:
from __future__ import annotations

from typing import Any, Dict, Optional, Tuple
import numpy as np


# ============================================================
# 1) Utilidades internas (inferencias y normalización)
# ============================================================

def _as_float_array(a: Any, *, name: str) -> np.ndarray:
    """Convierte a np.ndarray float64 y valida finitud."""
    arr = np.asarray(a, dtype=np.float64)
    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError(f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}")
    return arr


def _normalize_y_seq2one(y: np.ndarray, *, name: str = "y") -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).
    Acepta: (n,), (n,1). Rechaza shapes incompatibles.
    """
    if y.ndim == 1:
        return y
    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)
    raise ValueError(f"{name} shape inválido para seq2one. Se esperaba (n,) o (n,1). Recibido {y.shape}")


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiera (seq_len, n_features, mode) desde X.
    mode:
      - "3d": X=(n, seq_len, n_features)
      - "2d": X=(n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"
    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"
    raise ValueError(f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}")

In [17]:
# ============================================================
# 2) Sanity check principal (seq2one)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como "d_flat" esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado). Ej: 60*20=1200.
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len) (caso raro), permite tomar y[:, -1].
        Por defecto False (recomendado).
    """
    X = _as_float_array(X, name=f"X[{split_name}]")
    y = _as_float_array(y, name=f"y[{split_name}]")

    # Normalizar y
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        # caso tolerante: y=(n,seq_len) -> tomar último
        y = y[:, -1]
    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # Inferir modo y dimensiones de X
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # Validaciones básicas n_samples
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # Validación de shapes según modo
    if mode == "3d":
        # expected_seq_len / expected_n_features
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, recibido={seq_len}. X.shape={X.shape}"
            )
        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        # Si expected_flat_dim está, valida contra eso
        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, recibido={d_flat}. X.shape={X.shape}"
            )

        # Si no hay expected_flat_dim pero sí expected_seq_len, úselo como d_flat esperado
        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, recibido={d_flat}. X.shape={X.shape}"
            )

        # expected_n_features no aplica en 2D
        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim (ej: 1200) o pase X en 3D."
            )

    # Validación extra: varianza de y (para detectar targets constantes)
    y_std = float(np.std(y))
    if verbose:
        info = {
            "split": split_name,
            "X_shape": tuple(X.shape),
            "y_shape": tuple(y.shape),
            "mode": mode,
            "seq_len": seq_len if mode == "3d" else None,
            "n_features": n_features if mode == "3d" else None,
            "flat_dim": int(X.shape[1]) if mode == "2d" else None,
            "y_mean": float(np.mean(y)),
            "y_std": y_std,
            "y_min": float(np.min(y)),
            "y_max": float(np.max(y)),
        }
        print(
            f"[sanity_check_seq2one] {split_name} | X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | y_std={y_std:.6f}"
        )

    return {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "y_mean": float(np.mean(y)),
        "y_std": float(np.std(y)),
        "y_min": float(np.min(y)),
        "y_max": float(np.max(y)),
    }

In [18]:
# ============================================================
# 3) Sanity checks por bundle (train/valid/test)
# ============================================================

def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "horizon": 60,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Detecta automáticamente si X es 2D o 3D, y usa TRAIN como referencia
    si no se pasan expected_*.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    # Setear esperados desde TRAIN si no se dieron
    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        # En 3D no hace falta expected_flat_dim
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        # Para evitar confusión, no usamos expected_seq_len/n_features en 2D
        expected_seq_len = expected_seq_len  # puede quedar None
        expected_n_features = None

    # Ejecutar checks
    out_tr = sanity_check_seq2one(
        X_tr, y_tr, f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_va = sanity_check_seq2one(
        X_va, y_va, f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_te = sanity_check_seq2one(
        X_te, y_te, f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    h = bundle.get("horizon", "NA")
    if verbose:
        print(f"OK {tag} (h={h})")

    return {"train": out_tr, "valid": out_va, "test": out_te, "horizon": h}


def run_sanity_checks_all_horizons_seq2one(
    bundle_60: Dict[str, Any],
    bundle_90: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """Corre sanity checks para ambos horizontes (ej: 60 y 90)."""
    out_60 = run_sanity_checks_for_bundle_seq2one(bundle_60, tag="h60", verbose=verbose)
    out_90 = run_sanity_checks_for_bundle_seq2one(bundle_90, tag="h90", verbose=verbose)
    return {"h60": out_60, "h90": out_90}

In [19]:
#summary_delta = run_sanity_checks_all_horizons_seq2one(bundle_delta_60, bundle_delta_90)
#summary_ret = run_sanity_checks_all_horizons_seq2one(bundle_ret_60, bundle_ret_90)

# **DEFINICIÓN DE MODELO**

## **8. Definición del modelo — placeholder**

### **8.1. Modelo Ridge Regression (seq2one)**

**Idea básica**

**Ridge Regression** es una regresión lineal con **regularización L2**.  
Aprende un vector de pesos **w** que relaciona linealmente la entrada aplanada
con el target escalar, penalizando pesos grandes para reducir el sobreajuste.

Formalmente:

$$
\hat{y}_t = \mathbf{w}^\top \mathbf{x}_t + b
$$

con función objetivo:

$$
\min_{\mathbf{w}, b}
\sum_t (y_t - \hat{y}_t)^2
\;+\;
\alpha \sum_i w_i^2
$$

donde $\alpha\$ controla la **fuerza de la regularización**.

---

**Regularización (Ridge / Lasso)**

- **Ridge (L2):**
  - Penaliza el cuadrado de los coeficientes.
  - Reduce la magnitud de los pesos sin anularlos.
- **Lasso (L1):**
  - Penaliza el valor absoluto de los coeficientes.
  - Puede llevar pesos exactamente a cero (sparsity).

**Riesgo:** Bajo, controlado por diseño.  
La regularización es **intrínseca al modelo** y está gobernada por el
hiperparámetro \(\alpha\).

No se requieren técnicas adicionales como **dropout** o **early stopping**,
ya que no se trata de un modelo iterativo por épocas.

---

**Por qué Ridge encaja bien en este proyecto**

- Entrada de **alta dimensión**: 60 × 20 = **1200 features**.
- Features **altamente correlacionadas** (estructura temporal).
- Modelo:
  - simple,
  - estable,
  - rápido de entrenar,
  - interpretable como referencia lineal.

Ridge actúa como el **baseline entrenable** contra el cual se comparan
modelos más complejos (MLP, LSTM, TCN, Transformer).

---

**Hiperparámetros iniciales**

Para este stage (sin tuning):

- `alpha`: fijo (por ejemplo, `1.0`)
- `fit_intercept`: `True`
- `random_state`: no aplica
- **Sin validación interna** (la evaluación se realiza externamente en VALID)

El ajuste fino del coeficiente de regularización se aborda en etapas posteriores.


### **8.2. Implementación del modelo**

In [20]:
from sklearn.linear_model import Ridge

def build_ridge_model(*, alpha: float = 1.0) -> Ridge:
    """
    Construye un modelo Ridge Regression para seq2one.

    Parámetros
    ----------
    alpha : float
        Fuerza de regularización L2.

    Retorna
    -------
    model : sklearn.linear_model.Ridge
        Modelo Ridge configurado.
    """
    model = Ridge(
        alpha=alpha,
        fit_intercept=True,
    )
    return model


### **8.3. Entrenamiento (TRAIN)**

In [21]:
def train_ridge_for_bundle(bundle, *, alpha: float = 1.0):
    X_train = bundle["train"]["X"]
    y_train = bundle["train"]["y"]

    model = build_ridge_model(alpha=alpha)
    model.fit(X_train, y_train)
    return model

In [22]:
#ridge_60 = train_ridge_for_bundle(bundle_60, alpha=1.0)
#ridge_90 = train_ridge_for_bundle(bundle_90, alpha=1.0)

## **9. Métricas ML**

In [23]:
import pandas as pd

def metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    horizon: int,
    window_size: int,
    target: str,
) -> pd.DataFrame:
    """
    Convierte un dict de métricas seq2one en una fila de DataFrame.
    """

    return pd.DataFrame([{
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon_min": horizon,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics.get("R2"),
        "DA": metrics.get("DA"),
    }])

In [24]:
def get_metrics (bundle, model):

    # -------- VALID --------
    X_valid = bundle ['valid']['X']
    y_pred_valid = model.predict(X_valid)

    # --------TEST --------
    X_test = bundle ['test']['X']
    y_pred_test = model.predict(X_test)

    y_valid = bundle["valid"]["y"]
    y_test  = bundle["test"]["y"]

    metrics_valid = compute_seq2one_metrics(y_valid, y_pred_valid, compute_r2=True)
    metrics_test  = compute_seq2one_metrics(y_test,  y_pred_test,  compute_r2=True)

    return metrics_valid, metrics_test

## **10. Gestión de dataset de métricas**

In [25]:
def load_seq2one_metrics_if_exists(
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> pd.DataFrame:
    path = Path(base_dir) / f"seq2one_{name}_metrics.parquet"
    if path.exists():
        return pd.read_parquet(path)
    return pd.DataFrame()

In [26]:
from pathlib import Path
import pandas as pd

def save_seq2one_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas seq2one en formato Parquet.

    Parámetros
    ----------
    df_metrics : pd.DataFrame
        DataFrame con métricas (una fila por modelo/horizonte/split).
    name : str
        Nombre identificador del archivo (ej: 'naive_valid', 'ridge_valid').
        No incluir extensión.
    base_dir : str
        Directorio base donde se almacenan todas las métricas seq2one.

    Retorna
    -------
    out_path : Path
        Ruta completa del archivo guardado.
    """
    out_dir = Path(base_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"seq2one_{name}_metrics.parquet"
    df_metrics.to_parquet(out_path, index=False)

    print(f"[OK] Métricas guardadas en: {out_path}")
    return out_path

## **11. Ejecución completa**

In [27]:
import pandas as pd
import gc

def run_ridge(window_size: int, *, alpha: float = 1.0, verbose: bool = True):

    size = window_size

    if verbose:
        print("\n" + "=" * 80)
        print(f"RIDGE | SEQ2ONE | WINDOW_SIZE=L{size} | alpha={alpha}")
        print("=" * 80)

    targets = ["delta_60", "delta_90", "ret_60", "ret_90"]
    rows = []

    for target in targets:
        if verbose:
            print(f"\n[BUILD] L{size} | target = '{target}'")

        # crear SOLO 1 bundle (y aplanar X para Ridge)
        (bundle,) = create_bundles(
            window_size=size,
            targets=[target],            # <- SOLO UNO
            windows_paths=windows_paths,
            scalers_paths=scalers_paths,
            flatten_X=True,              # <- Ridge necesita 2D
        )

        if verbose:
            print(f"[TRAIN] L{bundle['window_size']} | target={bundle['target']} | ridge alpha={alpha}")

        # entrenar
        model = train_ridge_for_bundle(bundle, alpha=alpha)

        #  liberar TRAIN inmediatamente (ahorra RAM)
        del bundle["train"]
        gc.collect()

        # métricas (valid/test)
        metrics_valid, metrics_test = get_metrics(bundle, model)

        # a DF (VALID)
        rows.append(
            metrics_to_df(
                metrics_valid,
                model="ridge",
                split="valid",
                horizon=bundle["horizon"],
                window_size=bundle["window_size"],
                target=bundle["target"],
            )
        )

        # a DF (TEST)
        rows.append(
            metrics_to_df(
                metrics_test,
                model="ridge",
                split="test",
                horizon=bundle["horizon"],
                window_size=bundle["window_size"],
                target=bundle["target"],
            )
        )

        # liberar VALID/TEST + modelo + bundle
        del bundle
        del model
        del metrics_valid, metrics_test
        gc.collect()

    df_ridge_metrics = (
        pd.concat(rows, ignore_index=True)
          .sort_values(["window_size", "target", "split", "horizon_min", "model"])
          .reset_index(drop=True)
    )

    if verbose:
        print(f"\n[DONE] L{size} | rows={len(df_ridge_metrics)}")
        print(df_ridge_metrics[["window_size", "target", "split", "horizon_min", "model"]]
              .drop_duplicates()
              .to_string(index=False))

    return df_ridge_metrics

In [28]:
def run_ridge_incremental(
    window_sizes: list[int],
    *,
    alpha: float = 1.0,
    name: str = "ridge",  # -> seq2one_ridge_metrics.parquet
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
    verbose: bool = True,
) -> pd.DataFrame:

    # 1) Cargar si existe
    df_all = load_seq2one_metrics_if_exists(name=name, base_dir=base_dir)

    # 2) Asegurar columna alpha
    if df_all.empty:
        df_all = pd.DataFrame(columns=[
            "model","split","window_size","target","horizon_min","MAE","RMSE","R2","DA","alpha"
        ])
    if "alpha" not in df_all.columns:
        df_all["alpha"] = pd.NA

    key_cols = ["model","alpha","window_size","target","split","horizon_min"]

    # 3) Normalizar tipos (evita falsos mismatches)
    if len(df_all):
        df_all["window_size"] = pd.to_numeric(df_all["window_size"], errors="coerce").astype("Int64")
        df_all["horizon_min"] = pd.to_numeric(df_all["horizon_min"], errors="coerce").astype("Int64")

    # 4) Loop por window_size
    for ws in window_sizes:

        # Si ya existen todas las filas esperadas para este ws, saltar
        # Esperadas: 4 targets x 2 splits (valid/test) = 8 filas para ridge (siempre)
        # (Esto asume que tu run_ridge devuelve valid+test para los 4 targets)
        df_ws = df_all[(df_all["model"] == "ridge") & (df_all["alpha"] == alpha) & (df_all["window_size"] == ws)]
        if len(df_ws) >= 8:
            if verbose:
                print(f"[SKIP] L{ws}: ya hay {len(df_ws)} filas (ridge alpha={alpha}).")
            continue

        if verbose:
            print("\n" + "="*90)
            print(f"[RUN] RIDGE incremental | L{ws} | alpha={alpha}")
            print("="*90)

        # 5) Entrenar y obtener métricas para ESTE ws
        df_new = run_ridge(ws, alpha=alpha, verbose=verbose).copy()
        df_new["alpha"] = alpha  # agregar alpha

        # 6) Filtrar filas ya existentes (anti-duplicados)
        # Creamos un "key" para comparar rápido
        existing_keys = set(tuple(x) for x in df_all[key_cols].dropna().values)
        mask_keep = [tuple(row) not in existing_keys for row in df_new[key_cols].values]
        df_new = df_new.loc[mask_keep].copy()

        if df_new.empty:
            if verbose:
                print(f"[INFO] L{ws}: no había filas nuevas para agregar.")
            continue

        # 7) Merge + dedupe por seguridad
        df_all = pd.concat([df_all, df_new], ignore_index=True)
        df_all = df_all.drop_duplicates(subset=key_cols, keep="last").reset_index(drop=True)

        # 8) Guardar checkpoint con TU función
        save_seq2one_metrics(df_all, name=name, base_dir=base_dir)

        if verbose:
            print(f"[OK] Checkpoint guardado. Total rows={len(df_all)}")

    return df_all

In [29]:
df_ridge_all_sizes = load_seq2one_metrics_if_exists(name="ridge", base_dir="/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics")

In [30]:
RIDGE_ALL_TRAIN = '''
df_ridge_all_sizes = run_ridge_incremental(
    window_sizes=[30, 60, 90, 120, 180],
    alpha=1.0,
    name="ridge",   # genera seq2one_ridge_metrics.parquet
    verbose=True,
)

df_ridge_all_sizes
'''

In [31]:
display(df_ridge_all_sizes)

,model,split,window_size,target,horizon_min,MAE,RMSE,R2,DA,alpha
0,ridge,test,30,delta_60,60,50.776878,78.184379,0.243375,0.669053,1.0
1,ridge,valid,30,delta_60,60,33.838246,47.726024,0.241522,0.664962,1.0
2,ridge,test,30,delta_90,90,63.053931,97.290046,0.214467,0.661518,1.0
3,ridge,valid,30,delta_90,90,41.497789,58.784074,0.235739,0.652690,1.0
4,ridge,test,30,ret_60,60,0.002514,0.004165,0.188181,0.664476,1.0
5,ridge,valid,30,ret_60,60,0.001882,0.002667,0.229354,0.660137,1.0
6,ridge,test,30,ret_90,90,0.003152,0.005248,0.131294,0.657321,1.0
7,ridge,valid,30,ret_90,90,0.002323,0.003313,0.208837,0.646646,1.0
8,ridge,test,60,delta_60,60,43.907051,68.582439,0.411186,0.749068,1.0
9,ridge,valid,60,delta_60,60,28.644124,41.457055,0.439158,0.750437,1.0


## **12. Observaciones**

**1. Impacto de L30 → L60**

El salto de desempeño entre `L30` y `L60` es estructural y consistente.

Ejemplo `delta_60` (TEST R²):

- L30 → 0.243  
- L60 → 0.411  

La mejora es significativa y se observa también en otros targets.

**Conclusión:**  
L30 es insuficiente como ventana para capturar la dinámica relevante del sistema.

---

**2. Impacto de L60 → L90**

En este tramo la mejora deja de ser clara.

Ejemplo `delta_60` (TEST R²):

- L60 → 0.411  
- L90 → 0.419  

La mejora es marginal.

Ejemplo `ret_60` (TEST R²):

- L60 → 0.299  
- L90 → 0.275  

Empeora.

Ejemplo `ret_90` (TEST R²):

- L60 → 0.213  
- L90 → 0.203  

También empeora.

**Conclusión:**  
L90 no aporta una mejora consistente respecto a L60, especialmente en retornos.

---

**3. Impacto de L90 → L120**

En este tramo vuelve a observarse una mejora clara, pero únicamente en los targets tipo delta.

Ejemplo `delta_60` (TEST R²):

- L90 → 0.419  
- L120 → 0.455  

Ejemplo `delta_90` (TEST R²):

- L90 → 0.365  
- L120 → 0.409  

Ambos muestran mejoras relevantes.

Sin embargo, en retornos:

`ret_60` (TEST R²):

- L90 → 0.275  
- L120 → 0.254  

`ret_90` (TEST R²):

- L90 → 0.203  
- L120 → 0.167  

En ambos casos el desempeño empeora.

---

**4. Patrón estructural observado**

- Para targets tipo delta
  - Aumentar la ventana mejora el desempeño hasta L120.  
  - Existe evidencia de que mayor contexto temporal aporta información útil.

- Para targets tipo retorno
  - No se observa mejora al aumentar la ventana más allá de L60.  
  - Existe indicio de saturación o incluso degradación por exceso de lags.

---

**5. Interpretación técnica**

Los resultados son coherentes con una interpretación financiera y estadística:

- El delta absoluto puede depender más del régimen previo y del contexto acumulado.
- El retorno porcentual es más cercano a una variable estacionaria y puede requerir menos memoria temporal.
- Ridge es un modelo lineal; aumentar la cantidad de lags incrementa la multicolinealidad.
- En retornos, ese exceso de dimensionalidad puede perjudicar la generalización.

---

**6. Sobre la imposibilidad de entrenar L180**

No poder entrenar L180 debido a restricciones de RAM no es crítico.

La tendencia observada sugiere:

- L60 → L90: mejora marginal.
- L90 → L120: mejora en delta, no en retornos.
- No hay evidencia de que L180 vaya a producir una mejora estructural significativa.
- El costo computacional crece de forma muy marcada.

Es razonable asumir que L180 tendría:

- Mejora leve en delta.
- Mayor deterioro en retornos.
- Mayor varianza.
- Costo computacional excesivo.

---

**7. Decisión estratégica recomendada**

Para modelos lineales:

- Usar L60 para retornos.
- Usar L120 para delta.

Descartar:

- L30 de forma definitiva.
- L90 como ventana intermedia no dominante.
- L180, salvo que existan recursos computacionales suficientes y una hipótesis fuerte que lo justifique.

---

**8. Resultado más relevante**

El mejor desempeño observado hasta el momento es:

`delta_60 – L120 – VALID R² ≈ 0.505`

Para un modelo lineal, este nivel de explicación es significativo.

Esto indica:

- La ingeniería de features es sólida.
- El target delta presenta estructura lineal explotable.
- Existe señal predictiva real en el sistema.

---

**9. Resumen comparativo**

| Window | Delta | Retornos | Veredicto |
|--------|--------|----------|-----------|
| 30     | Débil  | Débil    | Descartar |
| 60     | Fuerte | Mejor    | Sólido    |
| 90     | Similar a L60 | Peor | Opcional |
| 120    | Mejor delta | Peor retorno | Óptimo para delta |

---

**Conclusión General**

No es prematuro comenzar a podar tamaños de ventana.

Con la evidencia actual:

- Mantener L60 y L120.
- Descartar L30.
- Evaluar si L90 aporta algún beneficio adicional real.
- No insistir con L180 en modelos lineales bajo restricciones de memoria.

Este análisis permite reducir complejidad computacional sin sacrificar desempeño y enfocar el desarrollo posterior en configuraciones más prometedoras.
